# 2024 NCAA Tournament Bracket Visualization

Side-by-side comparison of **model predictions** vs **actual results**.
- Predicted bracket: deterministic picks from our Logistic Regression (23 features) model
- Actual bracket: real 2024 tournament outcomes from ESPN data
- Green = correct pick, Red = wrong pick on the predicted bracket

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import sys

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.prediction.bracket_simulator import BracketSimulator
from src.data_collection.gamelog_collector import normalize_espn_name

DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
MODELS_DIR = PROJECT_ROOT / 'models'

print('Ready')

Ready


## 1. Build Real 2024 Bracket from ESPN Data
Trace backward from Elite 8 to identify which 16 teams belong to each region.

In [2]:
espn = pd.read_csv(DATA_DIR / 'espn_tournament_2024.csv')

def get_winner(game):
    if game['home_score'] > game['away_score']:
        return game['home_team_name'], int(game['home_seed'])
    return game['away_team_name'], int(game['away_seed'])

# Map 1-seed ESPN names to their NCAA region names
ONE_SEED_TO_REGION = {
    # 2024
    'Houston Cougars': 'South',
    'Purdue Boilermakers': 'Midwest',
    'UConn Huskies': 'East',
    'North Carolina Tar Heels': 'West',
    # 2023
    'Alabama Crimson Tide': 'South',
    'Houston Cougars': 'South',  # 2023 South too
    'Kansas Jayhawks': 'West',
    'Purdue Boilermakers': 'Midwest',
}
# Override per year to handle collisions (Houston is South in both years)
YEAR_REGION_MAP = {
    2024: {
        'Houston Cougars': 'South',
        'Purdue Boilermakers': 'Midwest',
        'UConn Huskies': 'East',
        'North Carolina Tar Heels': 'West',
    },
    2023: {
        'Alabama Crimson Tide': 'South',
        'Houston Cougars': 'Midwest',
        'Kansas Jayhawks': 'West',
        'Purdue Boilermakers': 'East',
    },
}

def discover_regions(espn_df):
    """
    Discover the 4 tournament regions by tracing backward from Elite 8.
    
    Returns regions ordered so that regions[0] vs regions[1] and
    regions[2] vs regions[3] in the Final Four, matching the simulator's
    pairing convention.
    """
    r64 = espn_df[espn_df['round'] == 'R64']
    r32 = espn_df[espn_df['round'] == 'R32']
    r16 = espn_df[espn_df['round'] == 'R16']
    r8 = espn_df[espn_df['round'] == 'R8'].sort_values('game_id')
    
    # Step 1: discover which 16 teams belong to each E8 game (= region)
    raw_regions = []
    e8_winners = []
    for _, e8_game in r8.iterrows():
        region_teams = set()
        e8_teams = {e8_game['home_team_name'], e8_game['away_team_name']}
        region_teams.update(e8_teams)
        
        for _, g in r16.iterrows():
            w, _ = get_winner(g)
            if w in e8_teams:
                region_teams.update([g['home_team_name'], g['away_team_name']])
        
        s16_teams = set(region_teams)
        for _, g in r32.iterrows():
            w, _ = get_winner(g)
            if w in s16_teams:
                region_teams.update([g['home_team_name'], g['away_team_name']])
        
        r32_teams = set(region_teams)
        for _, g in r64.iterrows():
            w, _ = get_winner(g)
            if w in r32_teams:
                region_teams.update([g['home_team_name'], g['away_team_name']])
        
        team_list = []
        for t in region_teams:
            seed = None
            for _, g in r64.iterrows():
                if g['home_team_name'] == t:
                    seed = int(g['home_seed'])
                    break
                elif g['away_team_name'] == t:
                    seed = int(g['away_seed'])
                    break
            if seed is not None:
                team_list.append({'name': t, 'seed': seed})
        
        team_list.sort(key=lambda x: x['seed'])
        raw_regions.append(team_list)
        
        w, _ = get_winner(e8_game)
        e8_winners.append(w)
    
    # Step 2: figure out F4 pairings and reorder regions so
    # regions[0] vs regions[1] and regions[2] vs regions[3] in F4
    r4 = espn_df[espn_df['round'] == 'R4'].sort_values('game_id')
    
    # Map each E8 winner to their region index
    winner_to_ri = {}
    for ri, w in enumerate(e8_winners):
        winner_to_ri[w] = ri
    
    # Find which regions pair in each F4 game
    f4_pairs = []
    for _, game in r4.iterrows():
        teams_in_game = [game['home_team_name'], game['away_team_name']]
        pair = []
        for t in teams_in_game:
            if t in winner_to_ri:
                pair.append(winner_to_ri[t])
        if len(pair) == 2:
            f4_pairs.append(tuple(sorted(pair)))
    
    if len(f4_pairs) == 2:
        # Reorder: first pair becomes indices 0,1; second pair becomes 2,3
        order = [f4_pairs[0][0], f4_pairs[0][1], f4_pairs[1][0], f4_pairs[1][1]]
        regions = [raw_regions[i] for i in order]
    else:
        # Fallback: keep original order
        regions = raw_regions
    
    return regions

espn_regions = discover_regions(espn)

def get_region_names(espn_regions, year=None):
    """Get region names. Uses NCAA region name if year mapping exists, else 1-seed name."""
    names = []
    year_map = YEAR_REGION_MAP.get(year, {}) if year else {}
    for region in espn_regions:
        one_seed = [t for t in region if t['seed'] == 1]
        if one_seed:
            espn_name = one_seed[0]['name']
            if espn_name in year_map:
                names.append(year_map[espn_name])
            else:
                sr = normalize_espn_name(espn_name)
                names.append(sr.split()[0] if ' ' in sr else sr)
        else:
            names.append('?')
    return names

REGION_NAMES = get_region_names(espn_regions, year=2024)

for i, region in enumerate(espn_regions):
    one_seed = [t for t in region if t['seed'] == 1]
    print(f"{REGION_NAMES[i]:12s}: {len(region)} teams  (1-seed: {one_seed[0]['name'][:25] if one_seed else '?'})")

South       : 16 teams  (1-seed: Houston Cougars)
Midwest     : 16 teams  (1-seed: Purdue Boilermakers)
East        : 16 teams  (1-seed: UConn Huskies)
West        : 16 teams  (1-seed: North Carolina Tar Heels)


<cell_type>markdown</cell_type>## 2. Model Configs & Bracket Setup
Define all available models and convert ESPN names to SR names for the simulator.

In [3]:
# ── Model configurations ──────────────────────────────────────────
MODELS = {
    'LogReg 23f': {
        'model': 'logistic_regression_23f.pkl',
        'scaler': 'scaler_23f.pkl',
        'features': 'features_23.pkl',
    },
    'XGBoost 23f': {
        'model': 'xgboost_tuned_23f.pkl',
        'scaler': 'scaler_23f.pkl',
        'features': 'features_23.pkl',
    },
    'RandomForest 23f': {
        'model': 'random_forest_tuned_23f.pkl',
        'scaler': 'scaler_23f.pkl',
        'features': 'features_23.pkl',
    },
    'LogReg 7f': {
        'model': 'logistic_regression_7f.pkl',
        'scaler': 'scaler_7f.pkl',
        'features': 'features_7.pkl',
    },
    'XGBoost 7f': {
        'model': 'xgboost_tuned_7f.pkl',
        'scaler': 'scaler_7f.pkl',
        'features': 'features_7.pkl',
    },
    'RandomForest 7f': {
        'model': 'random_forest_tuned_7f.pkl',
        'scaler': 'scaler_7f.pkl',
        'features': 'features_7.pkl',
    },
}

YEARS = [2023, 2024]

def espn_to_sr_bracket(espn_regions):
    """Convert ESPN team names to SR names for the bracket simulator."""
    bracket = []
    for region in espn_regions:
        sr_region = []
        for team in region:
            sr_name = normalize_espn_name(team['name'])
            sr_region.append({'name': sr_name, 'seed': team['seed'], 'espn_name': team['name']})
        bracket.append(sr_region)
    return bracket

print(f"Defined {len(MODELS)} models x {len(YEARS)} years = {len(MODELS)*len(YEARS)} brackets")
for name in MODELS:
    print(f"  - {name}")

Defined 6 models x 2 years = 12 brackets
  - LogReg 23f
  - XGBoost 23f
  - RandomForest 23f
  - LogReg 7f
  - XGBoost 7f
  - RandomForest 7f


## 3. Build Bracket Trees for Drawing

In [4]:
MATCHUP_ORDER = [0, 15, 7, 8, 4, 11, 3, 12, 5, 10, 2, 13, 6, 9, 1, 14]

def shorten(name):
    """Shorten team name for bracket display."""
    table = {
        'Connecticut': 'UConn', 'North Carolina': 'UNC',
        'Michigan State': 'Mich St', 'Mississippi State': 'Miss St',
        'San Diego State': 'SDSU', 'South Dakota State': 'SD State',
        'Florida Atlantic': 'FAU', 'Western Kentucky': 'W Kentucky',
        'Long Beach State': 'Long Beach', 'Iowa State': 'Iowa St',
        'Colorado State': 'Colorado St', 'South Carolina': 'S Carolina',
        'Utah State': 'Utah St', 'James Madison': 'James Mad.',
        'Grand Canyon': 'Gr Canyon', 'Texas Christian': 'TCU',
        'Brigham Young': 'BYU', 'Alabama-Birmingham': 'UAB',
        'Saint Mary\'s (CA)': "St Mary's", 'Morehead State': 'Morehead St',
        'Saint Peter\'s': "St Peter's", 'Washington State': 'Wash St',
        'North Carolina State': 'NC State', 'Texas A&M': 'Texas A&M',
        'Virginia Commonwealth': 'VCU', 'Florida Atlantic': 'FAU',
    }
    return table.get(name, name)


def get_r64_ordered(bracket):
    """Get R64 teams in standard matchup order per region."""
    r64 = {}
    for ri, region in enumerate(bracket):
        region_sorted = sorted(region, key=lambda x: x['seed'])
        ordered = []
        for i in range(0, 16, 2):
            ordered.append(region_sorted[MATCHUP_ORDER[i]])
            ordered.append(region_sorted[MATCHUP_ORDER[i + 1]])
        r64[ri] = ordered
    return r64


def build_pred_tree(results, bracket):
    """Build tree from simulator results."""
    tree = {'R64_teams': get_r64_ordered(bracket)}
    rounds_cfg = [('R64', 8), ('R32', 4), ('S16', 2), ('E8', 1)]
    for rname, gpr in rounds_cfg:
        key = rname + '_winners'
        tree[key] = {}
        games = results[rname]
        for ri in range(4):
            start = ri * gpr
            tree[key][ri] = [
                {'name': g['winner'], 'seed': g['winner_seed']}
                for g in games[start:start + gpr]
            ]
    tree['F4_winners'] = [
        {'name': g['winner'], 'seed': g['winner_seed']} for g in results['F4']
    ]
    tree['Championship_winner'] = {
        'name': results['Championship'][0]['winner'],
        'seed': results['Championship'][0]['winner_seed']
    }
    return tree


def build_actual_tree(espn_df, bracket):
    """
    Build actual results tree aligned to the SAME bracket slot positions
    as the predicted tree.
    """
    r64_teams = get_r64_ordered(bracket)
    tree = {'R64_teams': r64_teams}
    
    sr_to_slot = {}
    espn_to_slot = {}
    for ri in range(4):
        for si, team in enumerate(r64_teams[ri]):
            sr_to_slot[team['name'].lower()] = (ri, si)
            if 'espn_name' in team:
                espn_to_slot[team['espn_name'].lower()] = (ri, si)
    
    def find_slot(espn_name):
        key = espn_name.lower()
        if key in espn_to_slot:
            return espn_to_slot[key]
        sr = normalize_espn_name(espn_name).lower()
        if sr in sr_to_slot:
            return sr_to_slot[sr]
        return None
    
    def to_team(espn_name, seed):
        return {'name': normalize_espn_name(espn_name), 'seed': seed}
    
    tree['R64_winners'] = {ri: [None]*8 for ri in range(4)}
    for _, game in espn_df[espn_df['round'] == 'R64'].iterrows():
        w_name, w_seed = get_winner(game)
        slot = find_slot(game['home_team_name']) or find_slot(game['away_team_name'])
        if slot is None: continue
        ri, si = slot
        tree['R64_winners'][ri][si // 2] = to_team(w_name, w_seed)
    
    tree['R32_winners'] = {ri: [None]*4 for ri in range(4)}
    for _, game in espn_df[espn_df['round'] == 'R32'].iterrows():
        w_name, w_seed = get_winner(game)
        slot = find_slot(game['home_team_name']) or find_slot(game['away_team_name'])
        if slot is None: continue
        ri, si = slot
        tree['R32_winners'][ri][(si // 2) // 2] = to_team(w_name, w_seed)
    
    tree['S16_winners'] = {ri: [None]*2 for ri in range(4)}
    for _, game in espn_df[espn_df['round'] == 'R16'].iterrows():
        w_name, w_seed = get_winner(game)
        slot = find_slot(game['home_team_name']) or find_slot(game['away_team_name'])
        if slot is None: continue
        ri, si = slot
        tree['S16_winners'][ri][(si // 2) // 2 // 2] = to_team(w_name, w_seed)
    
    tree['E8_winners'] = {ri: [None]*1 for ri in range(4)}
    for _, game in espn_df[espn_df['round'] == 'R8'].iterrows():
        w_name, w_seed = get_winner(game)
        slot = find_slot(game['home_team_name']) or find_slot(game['away_team_name'])
        if slot is None: continue
        ri, _ = slot
        tree['E8_winners'][ri][0] = to_team(w_name, w_seed)
    
    tree['F4_winners'] = [None, None]
    for _, game in espn_df[espn_df['round'] == 'R4'].iterrows():
        w_name, w_seed = get_winner(game)
        slot_h = find_slot(game['home_team_name'])
        slot_a = find_slot(game['away_team_name'])
        regions_in_game = set()
        if slot_h: regions_in_game.add(slot_h[0])
        if slot_a: regions_in_game.add(slot_a[0])
        if regions_in_game & {0, 1}:
            tree['F4_winners'][0] = to_team(w_name, w_seed)
        elif regions_in_game & {2, 3}:
            tree['F4_winners'][1] = to_team(w_name, w_seed)
    
    final = espn_df[espn_df['round'] == 'FINAL'].iloc[0]
    w_name, w_seed = get_winner(final)
    tree['Championship_winner'] = to_team(w_name, w_seed)
    
    return tree

print('Tree builders defined')

Tree builders defined


## 4. Draw Brackets

In [5]:
def compute_accuracy(pred_tree, actual_tree):
    """Compute round-by-round and overall bracket accuracy."""
    def same(a, b):
        return bool(a and b and a.get('name','').lower() == b.get('name','').lower())
    
    rounds = [
        ('R64',   'R64_winners',  32),  # 4 regions × 8 games
        ('R32',   'R32_winners',  16),  # 4 regions × 4 games
        ('S16',   'S16_winners',   8),  # 4 regions × 2 games
        ('E8',    'E8_winners',    4),  # 4 regions × 1 game
    ]
    
    stats = {}
    total_correct = 0
    total_games = 0
    
    for label, key, expected in rounds:
        correct = 0
        count = 0
        for ri in range(4):
            preds = pred_tree.get(key, {}).get(ri, [])
            actuals = actual_tree.get(key, {}).get(ri, [])
            for i, p in enumerate(preds):
                a = actuals[i] if i < len(actuals) else None
                count += 1
                if same(p, a):
                    correct += 1
        stats[label] = (correct, count)
        total_correct += correct
        total_games += count
    
    # F4
    f4_correct = 0
    for i, p in enumerate(pred_tree.get('F4_winners', [])):
        actuals = actual_tree.get('F4_winners', [])
        a = actuals[i] if i < len(actuals) else None
        if same(p, a):
            f4_correct += 1
    stats['F4'] = (f4_correct, 2)
    total_correct += f4_correct
    total_games += 2
    
    # Championship
    ch_correct = 1 if same(pred_tree.get('Championship_winner'),
                           actual_tree.get('Championship_winner')) else 0
    stats['Champ'] = (ch_correct, 1)
    total_correct += ch_correct
    total_games += 1
    
    stats['Overall'] = (total_correct, total_games)
    return stats


def draw_bracket(ax, tree, title, compare_tree=None):
    """
    Draw NCAA bracket on ax.  4 regions stacked vertically, rounds L→R.
    compare_tree: color green/red for correct/wrong picks.
    When a pick is wrong, the actual winner is shown below in blue.
    """
    # ── sizing ────────────────────────────────────────────────────────
    FS       = 14       # team name font
    FS_SM    = 10       # actual-winner annotation font
    FS_HDR   = 14       # round header
    FS_REG   = 16       # region label (vertical)
    FS_TITLE = 28       # title
    ACTUAL_CLR = '#1565c0'  # blue for actual winner annotations

    RH   = 16           # vertical slots per region
    GAP  = 1.5          # vertical gap between regions

    COL_W = 1.7         # column-to-column spacing
    TW    = 1.45        # text width offset (where connector starts)
    RX = [i * COL_W for i in range(6)]

    total_h = 4 * RH + 3 * GAP
    ystarts = [3*(RH+GAP), 2*(RH+GAP), (RH+GAP), 0]

    ax.set_xlim(-1.0, RX[5] + 2.5)
    ax.set_ylim(-3.5, total_h + 2.5)
    ax.axis('off')
    ax.set_title(title, fontsize=FS_TITLE, fontweight='bold', pad=16)

    # ── helpers ───────────────────────────────────────────────────────
    def lbl(t):
        return f"({t['seed']:>2}) {shorten(t['name'])}"

    def same(a, b):
        return bool(a and b and a.get('name','').lower() == b.get('name','').lower())

    def get_actual(rkey, idx, ri=None):
        if compare_tree is None:
            return None
        if rkey == 'Championship_winner':
            return compare_tree.get(rkey)
        if rkey == 'F4_winners':
            cmp = compare_tree.get(rkey, [])
            return cmp[idx] if idx < len(cmp) else None
        cmp = compare_tree.get(rkey, {}).get(ri, [])
        return cmp[idx] if idx < len(cmp) else None

    def is_correct(team, rkey, idx, ri=None):
        actual = get_actual(rkey, idx, ri)
        return same(team, actual)

    def colour(team, rkey, idx, ri=None):
        if compare_tree is None:
            return '#111111'
        return '#006400' if is_correct(team, rkey, idx, ri) else '#cc0000'

    def put(x, y, team, c='#111111', bold=False, fs_override=None):
        ax.text(x, y, lbl(team), fontsize=fs_override or FS, color=c,
                fontweight='bold' if bold else 'normal',
                va='center', fontfamily='monospace', clip_on=False)

    def put_with_actual(x, y, team, rkey, idx, ri=None, bold=False, fs_override=None):
        c = colour(team, rkey, idx, ri)
        put(x, y, team, c=c, bold=bold, fs_override=fs_override)
        if compare_tree is not None and not is_correct(team, rkey, idx, ri):
            actual = get_actual(rkey, idx, ri)
            if actual:
                ax.text(x, y - 0.42, f"  actual: {lbl(actual)}",
                        fontsize=FS_SM, color=ACTUAL_CLR, fontstyle='italic',
                        va='center', fontfamily='monospace', clip_on=False)

    def bracket_line(x1, y1, y2, x2, yo):
        mx = (x1 + x2) / 2
        kw = dict(color='#888888', lw=1.0, solid_capstyle='round')
        ax.plot([x1, mx], [y1, y1], **kw)
        ax.plot([x1, mx], [y2, y2], **kw)
        ax.plot([mx, mx], [y1, y2], **kw)
        ax.plot([mx, x2], [yo, yo], **kw)

    # ── draw four regions ─────────────────────────────────────────────
    for ri in range(4):
        yb = ystarts[ri]

        ax.text(-0.7, yb + RH/2, REGION_NAMES[ri], fontsize=FS_REG,
                fontweight='bold', rotation=90, va='center', ha='center',
                color='#37474f')

        r64 = tree['R64_teams'][ri]
        for i, t in enumerate(r64):
            put(RX[0], yb + RH - 0.5 - i, t)

        w1 = tree.get('R64_winners', {}).get(ri, [])
        for i, t in enumerate(w1):
            y = yb + RH - 1 - i*2
            put_with_actual(RX[1], y, t, 'R64_winners', i, ri)
            bracket_line(RX[0]+TW, yb+RH-0.5 - i*2,
                                   yb+RH-1.5 - i*2, RX[1]-0.05, y)

        w2 = tree.get('R32_winners', {}).get(ri, [])
        for i, t in enumerate(w2):
            y = yb + RH - 2 - i*4
            put_with_actual(RX[2], y, t, 'R32_winners', i, ri)
            if len(w1) > i*2+1:
                bracket_line(RX[1]+TW, yb+RH-1 - (i*2)*2,
                                       yb+RH-1 - (i*2+1)*2, RX[2]-0.05, y)

        w3 = tree.get('S16_winners', {}).get(ri, [])
        for i, t in enumerate(w3):
            y = yb + RH - 4 - i*8
            put_with_actual(RX[3], y, t, 'S16_winners', i, ri)
            if len(w2) > i*2+1:
                bracket_line(RX[2]+TW, yb+RH-2 - (i*2)*4,
                                       yb+RH-2 - (i*2+1)*4, RX[3]-0.05, y)

        w4 = tree.get('E8_winners', {}).get(ri, [])
        if w4:
            y = yb + RH/2
            put_with_actual(RX[4], y, w4[0], 'E8_winners', 0, ri, bold=True)
            if len(w3) >= 2:
                bracket_line(RX[3]+TW, yb+RH-4, yb+RH-12, RX[4]-0.05, y)

    # ── Final Four ────────────────────────────────────────────────────
    f4    = tree.get('F4_winners', [])
    y_top = (ystarts[0] + RH/2 + ystarts[1] + RH/2) / 2
    y_bot = (ystarts[2] + RH/2 + ystarts[3] + RH/2) / 2

    for idx, yy, regions in [(0, y_top, [0,1]), (1, y_bot, [2,3])]:
        if idx < len(f4):
            put_with_actual(RX[5], yy, f4[idx], 'F4_winners', idx, bold=True)
            for pri in regions:
                if tree.get('E8_winners', {}).get(pri):
                    bracket_line(RX[4]+TW, ystarts[pri]+RH/2,
                                           ystarts[pri]+RH/2, RX[5]-0.05, yy)

    # ── Champion ──────────────────────────────────────────────────────
    ch = tree.get('Championship_winner')
    if ch and len(f4) >= 2:
        yc = (y_top + y_bot) / 2
        ax.text(RX[5]+1.2, yc+1.8, 'CHAMPION', fontsize=16, fontweight='bold',
                ha='center', va='bottom', color='#37474f')
        put_with_actual(RX[5]+0.2, yc, ch, 'Championship_winner', 0,
                        bold=True, fs_override=FS+3)
        bracket_line(RX[5]+TW, y_top, y_bot, RX[5]+0.15, yc)

    # ── Round headers ─────────────────────────────────────────────────
    hdr_y = max(ystarts) + RH + 1.5
    for i, h in enumerate(['R64', 'R32', 'Sweet 16', 'Elite 8', 'Final 4', 'Champ']):
        ax.text(RX[i]+0.6, hdr_y, h, fontsize=FS_HDR, ha='center',
                color='#546e7a', fontweight='bold')

    # ── Accuracy summary ──────────────────────────────────────────────
    if compare_tree is not None:
        stats = compute_accuracy(tree, compare_tree)
        ov_c, ov_t = stats['Overall']
        pct = 100 * ov_c / ov_t
        ax.text(RX[0], -2.8, f"Overall Accuracy: {pct:.1f}%",
                fontsize=18, fontfamily='monospace', fontweight='bold',
                color='#37474f', va='center', ha='left', clip_on=False)

print('draw_bracket() + compute_accuracy() defined')

draw_bracket() + compute_accuracy() defined


In [6]:
def generate_bracket_image(year, model_name='LogReg 23f', model_cfg=None):
    """Generate bracket visualization for a given tournament year and model.
    
    Args:
        year: Tournament year (e.g. 2023, 2024)
        model_name: Display name for the model (used in title and filename)
        model_cfg: Dict with 'model', 'scaler', 'features' filenames.
                   If None, uses MODELS[model_name].
    
    Returns:
        Overall accuracy percentage.
    """
    if model_cfg is None:
        model_cfg = MODELS[model_name]
    
    espn_yr = pd.read_csv(DATA_DIR / f'espn_tournament_{year}.csv')
    espn_regions_yr = discover_regions(espn_yr)
    region_names = get_region_names(espn_regions_yr, year=year)
    bracket_yr = espn_to_sr_bracket(espn_regions_yr)
    
    stats_path = DATA_DIR / f'sportsref_pretourney_{year}.csv'
    if not stats_path.exists():
        stats_path = DATA_DIR / f'sportsref_combined_{year}.csv'
    
    sim_yr = BracketSimulator(
        str(MODELS_DIR / model_cfg['model']),
        str(MODELS_DIR / model_cfg['scaler']),
        str(MODELS_DIR / model_cfg['features'])
    )
    sim_yr.load_team_stats(str(stats_path))
    predicted_yr = sim_yr.generate_deterministic_bracket(bracket_yr)
    
    pred_tree_yr = build_pred_tree(predicted_yr, bracket_yr)
    actual_tree_yr = build_actual_tree(espn_yr, bracket_yr)
    
    # Compute accuracy
    stats = compute_accuracy(pred_tree_yr, actual_tree_yr)
    ov_c, ov_t = stats['Overall']
    pct = 100 * ov_c / ov_t
    
    champ_pred = pred_tree_yr['Championship_winner']['name']
    champ_actual = actual_tree_yr['Championship_winner']['name']
    print(f"  {model_name} | {year} | {pct:.1f}% | Pred: {champ_pred} | Actual: {champ_actual}")
    
    # Draw
    fig_yr, ax_yr = plt.subplots(1, 1, figsize=(20, 28))
    fig_yr.patch.set_facecolor('white')
    
    global REGION_NAMES
    old_names = REGION_NAMES
    REGION_NAMES = region_names
    
    title = f'{model_name} — {year}'
    draw_bracket(ax_yr, pred_tree_yr, title, compare_tree=actual_tree_yr)
    
    REGION_NAMES = old_names
    
    legend_elements = [
        mpatches.Patch(facecolor='#e8f5e9', edgecolor='#006400', label='Correct Pick', linewidth=2),
        mpatches.Patch(facecolor='#ffebee', edgecolor='#cc0000', label='Wrong Pick', linewidth=2),
        mpatches.Patch(facecolor='#e3f2fd', edgecolor='#1565c0', label='Actual Winner', linewidth=2),
    ]
    ax_yr.legend(handles=legend_elements, loc='lower right', fontsize=16, frameon=True)
    
    # Sanitize model name for filename
    safe_name = model_name.lower().replace(' ', '_')
    fname = f'bracket_{year}_{safe_name}.png'
    out_path = PROJECT_ROOT / 'results' / 'visualizations' / fname
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close(fig_yr)
    
    return {'model': model_name, 'year': year, 'accuracy': pct,
            'champion_pred': champ_pred, 'champion_actual': champ_actual,
            'stats': stats}


# ── Generate all brackets ─────────────────────────────────────────
results = []
for year in YEARS:
    print(f"\n=== {year} ===")
    for model_name, model_cfg in MODELS.items():
        r = generate_bracket_image(year, model_name, model_cfg)
        results.append(r)

# ── Summary table ─────────────────────────────────────────────────
print("\n" + "="*70)
print(f"{'Model':<20s} {'Year':>4s}  {'Accuracy':>8s}  {'Predicted Champion':<25s}")
print("-"*70)
for r in results:
    correct = "✓" if r['champion_pred'].lower() == r['champion_actual'].lower() else ""
    print(f"{r['model']:<20s} {r['year']:>4d}  {r['accuracy']:>7.1f}%  {r['champion_pred']:<25s} {correct}")
print("="*70)


=== 2023 ===


Loaded stats for 364 teams
  LogReg 23f | 2023 | 50.8% | Pred: Gonzaga | Actual: Connecticut


Loaded stats for 364 teams
  XGBoost 23f | 2023 | 50.8% | Pred: Purdue | Actual: Connecticut


Loaded stats for 364 teams


  RandomForest 23f | 2023 | 47.6% | Pred: Kentucky | Actual: Connecticut


...

Loaded stats for 362 teams
  LogReg 7f | 2024 | 66.7% | Pred: Connecticut | Actual: Connecticut


Loaded stats for 362 teams
  XGBoost 7f | 2024 | 52.4% | Pred: Tennessee | Actual: Connecticut


Loaded stats for 362 teams


  RandomForest 7f | 2024 | 52.4% | Pred: Houston | Actual: Connecticut



Model                Year  Accuracy  Predicted Champion       
----------------------------------------------------------------------
LogReg 23f           2023     50.8%  Gonzaga                   
XGBoost 23f          2023     50.8%  Purdue                    
RandomForest 23f     2023     47.6%  Kentucky                  
LogReg 7f            2023     55.6%  Gonzaga                   
XGBoost 7f           2023     47.6%  UCLA                      
RandomForest 7f      2023     50.8%  Purdue                    
LogReg 23f           2024     61.9%  Connecticut               ✓
XGBoost 23f          2024     54.0%  Houston                   
RandomForest 23f     2024     60.3%  Houston                   
LogReg 7f            2024     66.7%  Connecticut               ✓
XGBoost 7f           2024     52.4%  Tennessee                 
RandomForest 7f      2024     52.4%  Houston                   
